# 03 · Train the cross-encoder reranker on hard negatives, then evaluate (GPU)

**Settings:** Accelerator **GPU T4 x2** · Internet **On** · **Add Input → notebook 01's output AND notebook 02's output** · Save & Run All.

1. **Mine hard negatives** for 200K training queries with our dense retriever (exact search over 8.8M on the GPU):
   ranks 4–100 after removing the judged positive (the very top non-positives are often unjudged relevant passages).
2. **Train two rerankers** with grouped LCE loss (1 positive + 7 hard negatives per query, fp16 mixed precision):
   - `ce-l6-dense-ft`: continue training the public `cross-encoder/ms-marco-MiniLM-L-6-v2` on *our* retriever's negatives
   - `ce-l6-from-base`: train from the base `nreimers/MiniLM-L6-H384-uncased` (no reranking head) for comparison
3. **Evaluate** all three rerankers on BM25 and dense candidates (top 100), dev + DL19 + DL20, with paired significance tests.

Estimated runtime: 1.5–2.5 h.

In [ ]:
# Clone the project (outside /kaggle/working: code is not an output) and install it.
import os, subprocess, sys, time, json, glob, shutil
T0 = time.time()
REPO = os.environ.get("REPO", "https://github.com/Rishabh763/Search-engine-Neural-Ranking.git")
import socket
try:
    socket.gethostbyname("github.com")
except OSError:
    raise RuntimeError(
        "No internet in this notebook. In the right sidebar open Session options "
        "(or Settings) and switch Internet ON, then restart and run again. "
        "The toggle only appears once your Kaggle account is phone-verified "
        "(Settings > Phone verification).") from None
if not os.path.exists("/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/tmp/repo"], check=True)
if not os.path.exists("/tmp/repo/scripts/train_reranker.py"):
    raise RuntimeError(
        "The repo on GitHub is older than this notebook (no scripts/train_reranker.py). "
        "Commit and push from your laptop, then delete /tmp/repo (!rm -rf /tmp/repo) and rerun.")
os.chdir("/tmp/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -e ".[dense]" 2>&1 | tail -3
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
def find_input(pattern):
    """Locate a file inside attached inputs (Kaggle mounts them under /kaggle/input/...)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"no input matches {pattern}: attach the previous notebook's output (Add Input)")
    return hits[0]

NB01 = find_input("data/processed/msmarco-full/manifest.json").split("/data/processed/")[0]
DENSE = os.path.dirname(find_input("indexes/msmarco-full/dense/*/meta.json"))
NB02 = DENSE.split("/indexes/")[0]
print("notebook 01:", NB01, "\nnotebook 02:", NB02)
# Read-only inputs are linked into a scratch index dir; everything new goes to /kaggle/working.
os.makedirs("/tmp/idx/msmarco-full/dense", exist_ok=True)
for src, dst in [(f"{NB02}/indexes/msmarco-full/docstore", "/tmp/idx/msmarco-full/docstore"),
                 (DENSE, f"/tmp/idx/msmarco-full/dense/{os.path.basename(DENSE)}")]:
    if not os.path.exists(dst):
        os.symlink(src, dst)
shutil.copytree(f"{NB02}/runs", "/kaggle/working/runs", dirs_exist_ok=True)
os.environ.update(DATA_DIR=f"{NB01}/data/processed", INDEX_DIR="/tmp/idx", RUNS_DIR="/kaggle/working/runs", PYTHONUTF8="1")
MODELS = "/kaggle/working/models"
NEG = "/kaggle/working/runs/msmarco-full/training/negatives-dense-exact.jsonl"
DENSE_TAG = "dense-exact-" + os.path.basename(DENSE)
!python -c "from searchengine.envinfo import environment_info, short_label; print(short_label(environment_info()))"

## 1. Hard negatives from our dense retriever

In [ ]:
!python scripts/mine_negatives.py --collection msmarco-full --source dense-exact --depth 100 --skip-top 3 --n-neg 15

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 2a. Continue training the public reranker on our negatives (small learning rate: it is already good)

In [ ]:
!python scripts/train_reranker.py --collection msmarco-full --negatives {NEG} --init cross-encoder/ms-marco-MiniLM-L-6-v2 --name ce-l6-dense-ft --models-dir {MODELS} --lr 7e-6 --batch-groups 32 --group-size 8

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 2b. Train from the base language model (randomly initialized reranking head)

In [ ]:
!python scripts/train_reranker.py --collection msmarco-full --negatives {NEG} --init nreimers/MiniLM-L6-H384-uncased --name ce-l6-from-base --models-dir {MODELS} --lr 3e-5 --batch-groups 32 --group-size 8

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 3. Rerank BM25 and dense candidates with all three rerankers (top 100; fp16 on the GPU)

In [ ]:
RERANKERS = ["cross-encoder/ms-marco-MiniLM-L-6-v2", f"{MODELS}/ce-l6-dense-ft", f"{MODELS}/ce-l6-from-base"]
for first in ["bm25-local", DENSE_TAG]:
    for m in RERANKERS:
        !python scripts/rerank.py --collection msmarco-full --first-stage {first} --depth 100 --model {m} --batch-size 128 2>&1 | grep -v "rerank@"

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 4. Is the fine-tuned reranker better? Paired randomization tests (dev: 6,980 queries; DL19+DL20 pooled: 97)

In [ ]:
PUB = "ms-marco-MiniLM-L-6-v2"
for first in ["bm25-local", DENSE_TAG]:
    for cand in ["ce-l6-dense-ft", "ce-l6-from-base"]:
        a, b = f"{first}+{PUB}@100", f"{first}+{cand}@100"
        print(f"\n=== {first}: public vs {cand} ===")
        !python scripts/compare_runs.py --collection msmarco-full --sets msmarco-dev --a "{a}" --b "{b}"
        !python scripts/compare_runs.py --collection msmarco-full --sets dl19 dl20 --a "{a}" --b "{b}"

## 5. Download bundle: models + all reports (small); full run files stay in the output if needed

In [ ]:
os.chdir("/kaggle/working")
with open("bundle_manifest.txt", "w") as f:
    for p in sorted(glob.glob("runs/msmarco-full/**/*.json", recursive=True)): f.write(p + "\n")
!zip -q -r results_bundle.zip models runs/msmarco-full/*/*.json runs/msmarco-full/training/*.stats.json bundle_manifest.txt
!ls -la /kaggle/working/results_bundle.zip

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)